In [ ]:
# ==============================================================================
# File: 04_Launch_Live_Demo.ipynb (Cell 1 of 3)
# What this notebook does: Mounts Google Drive, clones/syncs https://github.com/Alekhagrawal/ZeroFake.git,
#   installs all required dependencies, populates /content/ZeroFake/demo_samples/ with 4 real
#   FaceForensics++ test crops (Real Face, Known Fake, NeuralTextures Unseen Fake, and Blurry/Ambiguous Face),
#   and launches the interactive Gradio web dashboard (app.py) with a public *.gradio.live share URL.
# Inputs: /content/drive/MyDrive/ZeroFake_Storage/{cropped_faces.zip, checkpoints/zerofake/best_model.pt}.
# Outputs: Live public *.gradio.live URL for classroom presentation and interactive testing.
# ==============================================================================

# --- CELL 1: Mount Google Drive, Pull Latest GitHub Repo & Install Dependencies ---
import os
import shutil
import sys
from google.colab import drive

# 1. Safely clear any unmounted placeholder /content/drive directory before mounting
if not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')

# 2. Clone or force-sync https://github.com/Alekhagrawal/ZeroFake.git into /content/ZeroFake
GITHUB_REPO_URL = 'https://github.com/Alekhagrawal/ZeroFake.git'
REPO_DIR = '/content/ZeroFake'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print(f'[INFO] Fetching and syncing latest commits from {GITHUB_REPO_URL}...')
    !git -C /content/ZeroFake fetch origin
    !git -C /content/ZeroFake reset --hard origin/main
else:
    print(f'[INFO] Cloning {GITHUB_REPO_URL} into {REPO_DIR}...')
    !rm -rf /content/ZeroFake
    !git clone https://github.com/Alekhagrawal/ZeroFake.git /content/ZeroFake

%cd /content/ZeroFake
os.environ['PYTHONPATH'] = '/content/ZeroFake'
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

# 3. Install facenet-pytorch with --no-deps first (avoiding Pillow 10.2.0 compilation on Python 3.13)
#    and install remaining packages using 'grad-cam'
!pip install -q --no-deps facenet-pytorch
!pip install -q timm opencv-python-headless scikit-learn matplotlib seaborn grad-cam gradio pyyaml tqdm

# 4. Pre-flight CPU dry-run check on app.py
!PYTHONPATH=/content/ZeroFake python -u app.py --dry-run
if _exit_code != 0:
    raise RuntimeError(f'app.py --dry-run failed with exit code {_exit_code}')
print('[SUCCESS] Google Drive mounted, repo synced at /content/ZeroFake, and app.py --dry-run verified!')


In [ ]:
# --- CELL 2: Unzip cropped_faces.zip & Populate /content/ZeroFake/demo_samples/ ---
# Plain-English explanation:
#   Unzips cropped_faces.zip onto Colab's local SSD (/content/data) and automatically
#   selects 4 real test face images from the test splits:
#     1. sample_real_face.jpg        (1 Real test face from test_closed)
#     2. sample_known_fake.jpg       (1 Known Fake test face from Deepfakes/Face2Face/FaceSwap)
#     3. sample_unseen_fake.jpg      (1 Held-Out Unseen Generator Fake from NeuralTextures)
#     4. sample_blurry_ambiguous.jpg (1 heavily blurred/noisy face to trigger open-set UNKNOWN)
#   Saves them into /content/ZeroFake/demo_samples/ so all 4 one-click Gradio sample buttons work immediately.

import csv
import os
import shutil
import sys
import zipfile
import cv2
import numpy as np

%cd /content/ZeroFake
if '/content/ZeroFake' not in sys.path:
    sys.path.insert(0, '/content/ZeroFake')

from datasets.deepfake_dataset import DeepfakeDataset
from evaluation.robustness import apply_gaussian_blur_in_memory, apply_gaussian_noise_in_memory

STORAGE_DIR = '/content/drive/MyDrive/ZeroFake_Storage'
ZIP_PATH = os.path.join(STORAGE_DIR, 'cropped_faces.zip')
LOCAL_DATA_DIR = '/content/data'
SAMPLES_DIR = '/content/ZeroFake/demo_samples'
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(SAMPLES_DIR, exist_ok=True)

if os.path.exists(ZIP_PATH) and not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'crops')):
    print(f'[INFO] Unzipping {ZIP_PATH} onto local SSD {LOCAL_DATA_DIR}...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    print('[SUCCESS] Extracted cropped faces to /content/data.')

for fname in ('splits.json', 'frames_manifest.csv'):
    drive_file = os.path.join(STORAGE_DIR, fname)
    local_file = os.path.join(LOCAL_DATA_DIR, fname)
    if os.path.exists(drive_file) and not os.path.exists(local_file):
        shutil.copy2(drive_file, local_file)

# Flatten nested processed_faces/ folder if present
nested_dir = os.path.join(LOCAL_DATA_DIR, 'processed_faces')
if os.path.exists(nested_dir) and not os.path.exists(os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')):
    for item in os.listdir(nested_dir):
        src_item = os.path.join(nested_dir, item)
        dst_item = os.path.join(LOCAL_DATA_DIR, item)
        if not os.path.exists(dst_item):
            shutil.move(src_item, dst_item)

manifest_path = os.path.join(LOCAL_DATA_DIR, 'frames_manifest.csv')
splits_path = os.path.join(LOCAL_DATA_DIR, 'splits.json')

ds_closed = DeepfakeDataset(manifest_path=manifest_path, root_dir=LOCAL_DATA_DIR, split='test_closed', splits_path=splits_path, augment=False)
ds_unseen = DeepfakeDataset(manifest_path=manifest_path, root_dir=LOCAL_DATA_DIR, split='test_unseen', splits_path=splits_path, augment=False, include_closed_real_in_unseen=False)

real_src = next(s['image_path'] for s in ds_closed.samples if int(s['label']) == 0 and os.path.exists(s['image_path']))
known_fake_src = next(s['image_path'] for s in ds_closed.samples if int(s['label']) == 1 and os.path.exists(s['image_path']))
unseen_fake_src = next((s['image_path'] for s in ds_unseen.samples if os.path.exists(s['image_path'])), known_fake_src)

dst_real = os.path.join(SAMPLES_DIR, 'sample_real_face.jpg')
dst_known = os.path.join(SAMPLES_DIR, 'sample_known_fake.jpg')
dst_unseen = os.path.join(SAMPLES_DIR, 'sample_unseen_fake.jpg')
dst_ambig = os.path.join(SAMPLES_DIR, 'sample_blurry_ambiguous.jpg')

shutil.copy2(real_src, dst_real)
shutil.copy2(known_fake_src, dst_known)
shutil.copy2(unseen_fake_src, dst_unseen)

# Create heavily blurred + sensor-noisy ambiguous sample from unseen fake to demonstrate UNKNOWN rejection
bgr_unseen = cv2.imread(unseen_fake_src, cv2.IMREAD_COLOR)
rgb_unseen = cv2.cvtColor(bgr_unseen, cv2.COLOR_BGR2RGB)
blurred_rgb = apply_gaussian_blur_in_memory(rgb_unseen, ksize=5, sigma=2.0)
noisy_blurred_rgb = apply_gaussian_noise_in_memory(blurred_rgb, std=0.05)
cv2.imwrite(dst_ambig, cv2.cvtColor(noisy_blurred_rgb, cv2.COLOR_RGB2BGR))

print('[SUCCESS] Prepared all 4 one-click demo sample images inside /content/ZeroFake/demo_samples/:')
for name in sorted(os.listdir(SAMPLES_DIR)):
    print(f'  - {name}')


In [ ]:
# --- CELL 3: Launch Live ZeroFake Gradio Web App (Public *.gradio.live Share Link) ---
# Plain-English explanation:
#   Loads your trained Full ZeroFake checkpoint (with the validation-calibrated UNKNOWN
#   threshold tau_reject) and launches app.py with share=True, generating a live public
#   https://*.gradio.live link you can open on any laptop or projector browser.

%cd /content/ZeroFake
!PYTHONPATH=/content/ZeroFake python -u app.py --checkpoint /content/drive/MyDrive/ZeroFake_Storage/checkpoints/zerofake/best_model.pt --samples_dir /content/ZeroFake/demo_samples
